# Video Frames Extract

## Initialization

In [1]:
import cv2
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE_DIR = os.path.abspath("..")

INPUT_DIR = os.path.join(BASE_DIR, "data/WLBisindo/raw")
OUTPUT_DIR = os.path.join(BASE_DIR, "data/WLBisindo/frames")
CSV_PATH = os.path.join(BASE_DIR, "data/WLBisindo/classes.csv")

TARGET_FRAMES = 20

In [2]:
def load_label_map(csv_path):
    df = pd.read_csv(csv_path)
    return dict(zip(df["label"], df["gloss"]))

label_map = load_label_map(CSV_PATH)

print("Total kelas:", len(label_map))

Total kelas: 32


In [3]:
def get_label_from_filename(filename):
    parts = filename.split("_")
    for part in parts:
        if "label" in part:
            return int(part.replace("label", ""))
    return None

## Pre Process Data and Video

In [4]:
def is_motion(frame1, frame2, threshold=20000):
    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    return np.sum(gray) > threshold


def is_not_blurry(frame, threshold=50):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() > threshold


def uniform_sample(frames, num_samples):
    if len(frames) == 0:
        return []

    if len(frames) <= num_samples:
        return frames

    idxs = np.linspace(0, len(frames) - 1, num_samples).astype(int)
    return [frames[i] for i in idxs]

In [5]:
def process_video(video_path, output_folder):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < 5:
        return

    start = int(total_frames * 0.2)
    end = int(total_frames * 0.8)

    frames = []
    all_frames = []  # backup semua frame
    prev_frame = None

    cap.set(cv2.CAP_PROP_POS_FRAMES, start)

    for i in range(start, end):
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (224, 224))
        all_frames.append(frame)

        if prev_frame is None:
            prev_frame = frame
            continue

        if is_motion(prev_frame, frame) and is_not_blurry(frame):
            frames.append(frame)

        prev_frame = frame

    cap.release()

    # FALLBACK kalau filtering gagal
    if len(frames) < TARGET_FRAMES:
        frames = all_frames

    frames = uniform_sample(frames, TARGET_FRAMES)

    if len(frames) == 0:
        return

    os.makedirs(output_folder, exist_ok=True)

    for i, frame in enumerate(frames):
        cv2.imwrite(os.path.join(output_folder, f"frame_{i}.jpg"), frame)

In [6]:
def process_dataset():
    files = os.listdir(INPUT_DIR)

    print(f"Total files: {len(files)}")

    for file in tqdm(files, desc="Processing Videos"):
        if not file.endswith(".mp4"):
            continue

        label_id = get_label_from_filename(file)
        if label_id is None:
            print(f"Skip (no label): {file}")
            continue

        if label_id not in label_map:
            print(f"Skip (label not found): {file}")
            continue

        label_name = label_map[label_id]

        video_name = os.path.splitext(file)[0]

        output_folder = os.path.join(
            OUTPUT_DIR,
            label_name,
            video_name
        )

        video_path = os.path.join(INPUT_DIR, file)

        print(f"Processing: {file} → {label_name}")

        process_video(video_path, output_folder)

In [7]:
process_dataset()

Total files: 1600


Processing Videos:   0%|          | 0/1600 [00:00<?, ?it/s]

Processing: signer0_label0_sample1.mp4 → air


Processing Videos:   0%|          | 1/1600 [00:00<04:32,  5.87it/s]

Processing: signer0_label0_sample10.mp4 → air


...

Processing Videos: 100%|█████████▉| 1598/1600 [08:17<00:00,  3.61it/s]

Processing: signer4_label9_sample8.mp4 → saya


Processing Videos: 100%|█████████▉| 1599/1600 [08:17<00:00,  3.39it/s]

Processing: signer4_label9_sample9.mp4 → saya


Processing Videos: 100%|██████████| 1600/1600 [08:17<00:00,  3.21it/s]
